### Modelo de Regresion

***Nombre***: Gerard Almanzar
***Curso***: AI & ML
***Asignatura***: Modelos Supervisados

Acerca del dataset: propinas de un restaurante (244 filas, sin nulos). Cada fila es una cuenta de restaurante: cuánto fue el total (**total_bill**), cuánto dejaron de propina ( **tip** , lo que vamos a predecir), y datos de contexto: género de quien pagó, si fumaba, el día, si fue almuerzo o cena, y el tamaño del grupo.

***Objetivos de la Practica***: Crear un modelo de regresion lineal simple donde se pueda lograr predecir apartir de la variable **total_bill** cuanto se pago de propina usando la variable **tip** como la variable target.





In [44]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


### Carga del dataset y vista preliminar de los datos

In [3]:
df = pd.read_csv('../datasets/tips.csv')

In [29]:
print("Vista preliminar de los datos")
print('Dimension del dataset:', df.shape)

print("\nColumnas del dataset:")
print(list(df.columns))

print("\nTipo de datos del dataset:")
print(df.dtypes)

print("\nPrimeras Filas del dataset:")
print(df.head())

Vista preliminar de los datos
Dimension del dataset: (244, 7)

Columnas del dataset:
['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size']

Tipo de datos del dataset:
total_bill    float64
tip           float64
sex               str
smoker            str
day               str
time              str
size            int64
dtype: object

Primeras Filas del dataset:
   total_bill   tip     sex smoker  day    time  size
0       16.99  1.01  Female     No  Sun  Dinner     2
1       10.34  1.66    Male     No  Sun  Dinner     3
2       21.01  3.50    Male     No  Sun  Dinner     3
3       23.68  3.31    Male     No  Sun  Dinner     2
4       24.59  3.61  Female     No  Sun  Dinner     4


### Busqueda de datos nulos y Duplicados

In [22]:
print("Buscando valores nulos:")
print(df.isnull().sum())
print("\nTotal de datos duplicados:", df.duplicated(keep=False, ).sum())

duplicados_completos = df[df.duplicated(keep=False)]
print(duplicados_completos.sort_values(by=['sex'], ascending=False).head(20).to_string())

Buscando valores nulos:
total_bill    0
tip           0
sex           0
smoker        0
day           0
time          0
size          0
dtype: int64

Total de datos duplicados: 2
     total_bill  tip     sex smoker   day   time  size
198        13.0  2.0  Female    Yes  Thur  Lunch     2
202        13.0  2.0  Female    Yes  Thur  Lunch     2


### Interpretacion inicial.

Los hallazgos preliminares arrajoron 0 datos faltantes pero existen datos duplicados en la facturas '198' y '202'. Sin embargo, hay poca informacion para determinar si fue el mismo cliente o si dos clientes del mismo genero hicieron el mismo consumo en el restaurante. Por esta razon, se determina que los duplicados se van a quedar sin cambios.

### Fase de Entrenamiento del modelo regresion simple

In [82]:
X = df[['total_bill']]
y = df['tip']

modelo = LinearRegression()
modelo.fit(X,y)

print()
print('Pendiente:', round(modelo.coef_[0],2))
print('Intercepto:', round(modelo.intercept_,2))

print(df['day'].value_counts())
print()
print(df[df['day'] =='Thur'][['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size']].sort_values('total_bill', ascending=False).head(20))



Pendiente: 0.11
Intercepto: 0.92
day
Sat     87
Sun     76
Thur    62
Fri     19
Name: count, dtype: int64

     total_bill   tip     sex smoker   day    time  size
197       43.11  5.00  Female    Yes  Thur   Lunch     4
142       41.19  5.00    Male     No  Thur   Lunch     5
85        34.83  5.17  Female     No  Thur   Lunch     4
141       34.30  6.70    Male     No  Thur   Lunch     6
83        32.68  5.00    Male    Yes  Thur   Lunch     2
125       29.80  4.20  Female     No  Thur   Lunch     6
192       28.44  2.56    Male    Yes  Thur   Lunch     2
77        27.20  4.00    Male     No  Thur   Lunch     4
143       27.05  5.00  Female     No  Thur   Lunch     6
88        24.71  5.85    Male     No  Thur   Lunch     2
119       24.08  2.92  Female     No  Thur   Lunch     4
129       22.82  2.18    Male     No  Thur   Lunch     3
78        22.76  3.00    Male     No  Thur   Lunch     2
89        21.16  3.00    Male     No  Thur   Lunch     2
204       20.53  4.00    Male    Yes

### Interpretacion

Por cada 1 USD adicional en la cuenta total, el modelo espera que la propina suba, en promedio, 0.105 USD. Con una cuenta de 30 USD, el modelo predice una propina de $4.07.

### Modelo de Regresion Lineal Multiple: Todas las variables, incluyendo las de texto

In [54]:
valores_categoricos = df.select_dtypes(include=['str', 'object']).columns
valores_numericos =  df.select_dtypes(include=['int64', 'float64']).columns

print('\nColumnas Categoricas:',valores_categoricos)
print('Columnas Numericas:', valores_numericos)

#One-hot encoding
df_enc = pd.get_dummies(df, columns = valores_categoricos, drop_first=False)

# Separar X, y
X = df_enc.drop(columns=['tip'])
y = df_enc['tip']

#Dividir train/test

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

#Entrenar el modelo
modelo_multiple = LinearRegression()
modelo_multiple.fit(X_train, y_train)

print()
print(f"Columnas antes: {df.shape[1]} -> despues del encoding {df_enc.shape[1]}")
print(f"X_train: {X_train.shape} X_test: {X_test.shape}")

print("\nCoeficientes: ")
for nombre, coef in zip(X.columns, modelo_multiple.coef_):
    print(f"{nombre} -> {round(coef, 4)}")

print('\nIntercepto: ', round(modelo_multiple.intercept_, 4))



Columnas Categoricas: Index(['sex', 'smoker', 'day', 'time'], dtype='str')
Columnas Numericas: Index(['total_bill', 'tip', 'size'], dtype='str')

Columnas antes: 7 -> despues del encoding 13
X_train: (195, 12) X_test: (49, 12)

Coeficientes: 
total_bill -> 0.0947
size -> 0.2335
sex_Female -> -0.0144
sex_Male -> 0.0144
smoker_No -> 0.0962
smoker_Yes -> -0.0962
day_Fri -> 0.1041
day_Sat -> -0.0817
day_Sun -> 0.0533
day_Thur -> -0.0756
time_Dinner -> -0.0475
time_Lunch -> 0.0475

Intercepto:  0.5891


### Que tan bueno es el modelo?

***Metricas a utilizar***: MAE, MSE, RMSE, R2

In [64]:
y_pred = modelo.predict(X_test[['total_bill']])
y_pred_multiple = modelo_multiple.predict(X_test)

mae_simple = mean_absolute_error(y_test, y_pred)
mae_multiple = mean_absolute_error(y_test, y_pred_multiple)

mse_simple = mean_squared_error(y_test, y_pred)
mse_multiple = mean_squared_error(y_test, y_pred_multiple)

rmse_simple =  mse_simple ** 0.5
rmse_multiple =  mse_multiple ** 0.5

r2_simple = r2_score(y_test, y_pred)
r2_multiple = r2_score(y_test, y_pred_multiple)

print("Resultados del Modelo de Regresion Lineal Simple vs Multiple")
print("\nMetricas de Regresion Lineal Simple)")
print(f"MAE: {round(mae_simple, 4)} | MSE: {round(mse_simple, 4)} | RMSE: {round(rmse_simple, 4)} | R2 {round(r2_simple, 4)} ")
print("\nMetricas de Regresion Lineal Multiple)")
print(f"MAE: {round(mae_multiple, 4)} | MSE: {round(mse_multiple, 4)} | RMSE: {round(rmse_multiple, 4)} | R2 {round(r2_multiple, 4)} ")

Resultados del Modelo de Regresion Lineal Simple vs Multiple

Metricas de Regresion Lineal Simple)
MAE: 0.608 | MSE: 0.5491 | RMSE: 0.741 | R2 0.5607 

Metricas de Regresion Lineal Multiple)
MAE: 0.6671 | MSE: 0.7034 | RMSE: 0.8387 | R2 0.4373 


### Para Pensar

En la clase, el modelo múltiple de vivienda fue mejor que el simple. Aquí, con propinas, el modelo simple gana. Ambos resultados son
reales, no un error.

¿Qué diferencia hay entre los dos datasets que podría explicar por qué agregar variables no ayudó esta vez? Piensa en qué tan fuerte es la
relación real entre el día de la semana, o si alguien fuma, con el monto de la propina.

Respuesta: en el dataset de viviendas existia una relacion entre las variables ya que contenia propiedades o datos relacionados con la vivienda, sin embargo en este dataset dicha relacion no esta mostrando patrones o comportamientos entre los datos. No hay mucho que aprender de si el cliente fuma o segun el dia que haya ido al restaurante. Cada dia que el restaurante opera hay clientes consumiendo entre rangos de 50 los mas altos hasta debajo de 10 USD y son cliente que fuman y no fuman, pero que tambien son clientes en ambos generos. Por ende, que el modelo aprenda de dichas caracteristicas no va a encontrar ningun patron importante y esto hace que simplemente con una variable tenga mejor rendimiento.